# timeSpace — Reference Object Explorer (Colab)

Interactive [Stommel diagram](https://www.pmel.noaa.gov/foci/publications/2010/vanc0749.pdf)
of the **102 reference objects** shipped with
[timeSpace](https://github.com/MDunitz/timeSpace), spanning molecular to
planetary scales across 10 categories. Color indicates category.

**Add your own objects** in the "Add your own objects" cell, then run the
explorer cell to see them plotted alongside the reference set. Toggle
categories, pin an object, and hover any glyph for details.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MDunitz/timeSpace/blob/main/docs/reference_explorer_colab.ipynb)


In [ ]:
# Install dependencies and clone the package
!pip install -q astropy bokeh colorcet pandas
!git clone -q https://github.com/MDunitz/timeSpace.git
import sys; sys.path.insert(0, "timeSpace")

In [ ]:
import html
import pandas as pd
from IPython.display import HTML, display

from timeSpace.explorer import build_explorer, load_reference_objects
from timeSpace.explorer.config import CATEGORY_COLORS

REF_CSV = "timeSpace/data/datasets/time_space_reference_objects.csv"
VALID_CATEGORIES = sorted(CATEGORY_COLORS)
print("Valid categories:", VALID_CATEGORIES)

## Reference objects

The 102 reference objects ship with the repo. Schema:
`Name, Time_min, Time_max, Space_min, Space_max, Category, Reference`
(times in **seconds**, spaces in **m3**).


In [ ]:
reference = pd.read_csv(REF_CSV)
print(f"{len(reference)} reference objects across {reference.Category.nunique()} categories")
reference.groupby("Category").size().sort_values(ascending=False)

## Add your own objects

Edit `custom_objects` below. Each row needs:
- **Name** - shown on hover / as the pinned label
- **Time_min**, **Time_max** - characteristic time span in **seconds**
- **Space_min**, **Space_max** - characteristic volume span in **m3**
- **Category** - one of the 10 valid categories printed above

Leave the list empty to plot only the reference set.


In [ ]:
custom_objects = pd.DataFrame([
    # Uncomment and edit, or add your own rows:
    # {"Name": "Algae raceway pond", "Time_min": 1e4, "Time_max": 1e7,
    #  "Space_min": 1e0, "Space_max": 1e3, "Category": "Human-built",
    #  "Reference": "my data"},
])

if len(custom_objects):
    unknown = set(custom_objects["Category"]) - set(VALID_CATEGORIES)
    assert not unknown, f"Unknown categories {unknown}. Use one of {VALID_CATEGORIES}"
    combined = pd.concat([reference, custom_objects], ignore_index=True)
    print(f"Added {len(custom_objects)} custom object(s) -> {len(combined)} total")
else:
    combined = reference
    print("No custom objects added - plotting the 102 reference objects")

COMBINED_CSV = "combined_objects.csv"
combined.to_csv(COMBINED_CSV, index=False)

## Interactive explorer

Toggle categories with the checkboxes, pick an object to pin its label, and
hover any glyph for details. This is the same explorer published to the blog,
seeded with your rows.


In [ ]:
build_explorer(COMBINED_CSV, "my_explorer.html", mode="toggle")

# Render the standalone explorer inline (isolated iframe so BokehJS loads cleanly)
doc = open("my_explorer.html").read()
display(HTML(
    f'<iframe srcdoc="{html.escape(doc)}" '
    f'width="100%" height="720" style="border:1px solid #ddd;"></iframe>'
))

## Export

Both explorer modes are written to HTML you can download from the Colab file
browser (left sidebar) and embed via iframe:
- `my_explorer.html` - toggle mode (multi-category)
- `my_explorer_select.html` - select mode (one at a time + custom-object panel)


In [ ]:
build_explorer(COMBINED_CSV, "my_explorer_select.html", mode="select")
print("Saved my_explorer.html and my_explorer_select.html - download from the file browser")